## Import required libraries

In [1]:
# @title
%%capture
!pip install virtualenv
!git clone https://github.com/SalesforceAIResearch/uni2ts.git
%cd uni2ts
!pip install -e .

In [2]:
# @title
%%capture
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

**Restart your runtime session now, and then continue**





In [1]:
# @title
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from einops import rearrange
from gluonts.dataset.multivariate_grouper import MultivariateGrouper
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.eval_util.plot import plot_single, plot_next_multi
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.6.1, so it will not be used.
  warnings.warn(


In [2]:
!pip install sktime
import gc
import json
import os
from datetime import datetime
from matplotlib.dates import AutoDateLocator
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 26.9 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
CONFIG_PATH = '/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json'
OUTPUT_DIR = '/content/drive/MyDrive/results/json/Moirai'
MODEL_ID = 'Salesforce/moirai-1.1-R-small'
TRAIN_RATIO = 0.80
SHOW_PLOTS = True

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

DATASETS = CONFIG['datasets']
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Helper functions

In [5]:
from sklearn.preprocessing import MinMaxScaler


def scale_values(y_train, y_true, y_pred):
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1, 1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1, 1)

    scaler = MinMaxScaler()
    scaler.fit(y_train)
    y_true_scaled = scaler.transform(y_true)
    y_pred_scaled = scaler.transform(y_pred)

    return y_true_scaled, y_pred_scaled


In [6]:
def pinball_loss(q, y, alpha):
    return (alpha - (y < q)) * (y - q)


def crps_moirai_quantile_version(samples, y_true):
    samples = np.array(samples)
    y_true = np.array(y_true)

    alphas = np.arange(0.1, 1.0, 0.1)
    q_preds = {
        alpha: np.quantile(samples, alpha, axis=0)
        for alpha in alphas
    }

    denominator = np.sum(np.abs(y_true))
    wql_values = []

    for alpha in alphas:
        q = q_preds[alpha]
        pin = pinball_loss(q, y_true, alpha)
        wql = 2 * np.sum(pin) / denominator
        wql_values.append(wql)

    return float(np.mean(wql_values))


In [7]:
def save_timeseries_full(
    log_path, model_name, full_dates, full_values, train_dates, train_values,
    test_dates, test_values, forecast_dates, forecast_quantiles, metrics
):
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8') as file:
            log_data = json.load(file)
    else:
        log_data = []

    forecast_data = {'dates': [str(date) for date in forecast_dates]}
    for name, values in forecast_quantiles.items():
        forecast_data[name] = [float(value) for value in values]

    entry = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model': model_name,
        'full_series': {'dates': [str(date) for date in full_dates], 'values': [float(value) for value in full_values]},
        'train_series': {'dates': [str(date) for date in train_dates], 'values': [float(value) for value in train_values]},
        'test_series': {'dates': [str(date) for date in test_dates], 'values': [float(value) for value in test_values]},
        'forecast': forecast_data,
        'metrics': {
            name: None if not np.isfinite(value) else float(value)
            for name, value in metrics.items()
        },
    }

    log_data.append(entry)
    with open(log_path, 'w', encoding='utf-8') as file:
        json.dump(log_data, file, indent=4, ensure_ascii=False)


## Load the pretrained Moirai model

In [8]:
moirai_module = MoiraiModule.from_pretrained(MODEL_ID)


config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 55.3MB            

model.safetensors: downloading bytes:           |  0.00B            

## Prepare one target column

In [9]:
def prepare_target(data, dataset, target):
    dataset_name = dataset['dataset_name']
    time_column = dataset['time_column']
    target_column = target['column']

    target_data = data[[time_column, target_column]].copy()
    target_data[target_column] = pd.to_numeric(target_data[target_column])

    dates = target_data[time_column]
    values = target_data[target_column].to_numpy(dtype=float)

    return dataset_name, time_column, target_column, dates, values


## Split data into training and test parts

In [10]:
def split_data(dates, values):
    split_index = int(len(values) * TRAIN_RATIO)

    train_dates = dates.iloc[:split_index]
    train_values = values[:split_index]
    test_dates = dates.iloc[split_index:]
    test_values = values[split_index:]

    return train_dates, train_values, test_dates, test_values


## Create a Moirai forecast

In [11]:
def create_forecast(train_values, test_values):
    model = MoiraiForecast(
        module=moirai_module,
        prediction_length=len(test_values),
        context_length=len(train_values),
        patch_size=32,
        num_samples=100,
        target_dim=1,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
    )

    past_target = torch.as_tensor(train_values, dtype=torch.float32).reshape(1, -1, 1)
    past_observed_target = torch.ones_like(past_target, dtype=torch.bool)
    past_is_pad = torch.zeros_like(past_target, dtype=torch.bool).squeeze(-1)

    with torch.no_grad():
        forecast = model(
            past_target=past_target,
            past_observed_target=past_observed_target,
            past_is_pad=past_is_pad,
        )

    samples = forecast[0].detach().cpu().numpy()
    if samples.ndim == 3:
        samples = samples[..., 0]

    quantiles = {
        'p5': np.quantile(samples, 0.05, axis=0),
        'p10': np.quantile(samples, 0.10, axis=0),
        'p25': np.quantile(samples, 0.25, axis=0),
        'median': np.quantile(samples, 0.50, axis=0),
        'p75': np.quantile(samples, 0.75, axis=0),
        'p90': np.quantile(samples, 0.90, axis=0),
        'p95': np.quantile(samples, 0.95, axis=0),
    }

    del model, forecast, past_target, past_observed_target, past_is_pad
    gc.collect()
    return samples, quantiles


## Calculate forecast metrics

In [12]:
def calculate_metrics(train_values, test_values, predictions, samples):
    train_array = np.asarray(train_values, dtype=np.float64)
    test_array = np.asarray(test_values, dtype=np.float64)
    prediction_array = np.asarray(predictions, dtype=np.float64)

    mase = MeanAbsoluteScaledError()(
        test_array, prediction_array, y_train=train_array
    )

    # Unscaled errors use the original values.
    mae = float(mean_absolute_error(test_array, prediction_array))
    mse = float(mean_squared_error(test_array, prediction_array))
    rmse = float(np.sqrt(mse))

    # Scaled errors use a MinMaxScaler fitted only on the training values.
    test_scaled, predictions_scaled = scale_values(
        train_array, test_array, prediction_array
    )
    scaled_mae = float(mean_absolute_error(test_scaled, predictions_scaled))
    scaled_mse = float(mean_squared_error(test_scaled, predictions_scaled))
    scaled_rmse = float(np.sqrt(scaled_mse))

    metrics = {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mase': float(mase),
        'mae_scaled': scaled_mae,
        'mse_scaled': scaled_mse,
        'rmse_scaled': scaled_rmse,
        'crps': crps_moirai_quantile_version(samples, test_values),
    }

    return metrics


## Draw the forecast chart

In [13]:
def show_forecast(train_dates, train_values, test_dates, test_values, quantiles,
                  dataset_name, time_column, target_column):
    if not SHOW_PLOTS:
        return

    plt.figure(figsize=(16, 6))
    plt.plot(train_dates, train_values, label='Training data')
    plt.plot(test_dates, test_values, label='Actual values')
    plt.plot(test_dates, quantiles['median'], '--', label='Moirai median')
    plt.fill_between(test_dates, quantiles['p25'], quantiles['p75'], alpha=0.25, label='50% interval')
    plt.fill_between(test_dates, quantiles['p10'], quantiles['p90'], alpha=0.15, label='80% interval')
    plt.title(f'{dataset_name} - {target_column}')
    plt.xlabel(time_column)
    plt.ylabel(target_column)
    plt.legend()
    plt.grid()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    plt.close()


## Save one result as JSON

In [14]:
def save_result(dataset_name, target_column, target, dates, values,
                train_dates, train_values, test_dates, test_values,
                quantiles, metrics):
    output_file = target.get(
        'output_file', f'{dataset_name}_{target_column}.json'
    )
    log_path = os.path.join(OUTPUT_DIR, output_file)
    model_name = MODEL_ID

    save_timeseries_full(
        log_path,
        model_name,
        dates,
        values,
        train_dates,
        train_values,
        test_dates,
        test_values,
        test_dates,
        quantiles,
        metrics,
    )

    result = {
        'dataset': dataset_name,
        'target': target_column,
        'model': model_name,
        'output_file': output_file,
    }
    result.update(metrics)
    all_results.append(result)

    print('JSON saved:', log_path)


## Run all steps for one target column

In [15]:
def run_target(data, dataset, target, model_number):
    dataset_name, time_column, target_column, dates, values = prepare_target(
        data, dataset, target
    )

    train_dates, train_values, test_dates, test_values = split_data(dates, values)
    samples, quantiles = create_forecast(train_values, test_values)
    predictions = quantiles['median']
    metrics = calculate_metrics(train_values, test_values, predictions, samples)

    print(f"\nModel {model_number}/{total_models}: {dataset_name} - {target_column}")
    print('MAE scaled:', round(metrics['mae_scaled'], 4))
    print('RMSE scaled:', round(metrics['rmse_scaled'], 4))
    print('MASE:', round(metrics['mase'], 4))

    show_forecast(
        train_dates,
        train_values,
        test_dates,
        test_values,
        quantiles,
        dataset_name,
        time_column,
        target_column,
    )

    save_result(
        dataset_name,
        target_column,
        target,
        dates,
        values,
        train_dates,
        train_values,
        test_dates,
        test_values,
        quantiles,
        metrics,
    )


## Run all datasets and target columns

In [16]:
total_models = sum(len(dataset['targets']) for dataset in DATASETS)
all_results = []
completed_models = 0

for dataset in DATASETS:
    dataset_name = dataset['dataset_name']
    file_path = dataset['file_path']
    time_column = dataset['time_column']

    data = pd.read_csv(file_path)
    data[time_column] = pd.to_datetime(data[time_column])
    data = data.sort_values(time_column).reset_index(drop=True)

    print('\n' + '=' * 60)
    print('Dataset:', dataset_name)

    for target in dataset['targets']:
        completed_models += 1
        run_target(data, dataset, target, completed_models)


Output hidden; open in https://colab.research.google.com to view.

In [17]:
print('\n\n')
print('=' * 110)
print('ALL DATASETS / TARGETS COMPLETED')
print('=' * 110)
if len(all_results) > 0:
    summary_df = pd.DataFrame(all_results)
    print(summary_df.to_string(index=False))
else:
    print('⚠️ No target was successfully completed.')




ALL DATASETS / TARGETS COMPLETED
dataset         target                         model               output_file           mae          mse          rmse      mase  mae_scaled  mse_scaled  rmse_scaled     crps
   WB90             wt Salesforce/moirai-1.1-R-small                WB_wt.json      0.106406 3.090553e-02      0.175800  0.791726    0.039120    0.004177     0.064632 0.003408
   WB90            toc Salesforce/moirai-1.1-R-small               WB_toc.json      0.054483 4.545058e-03      0.067417  0.636644    0.115922    0.020575     0.143441 0.009044
   WB90    temperature Salesforce/moirai-1.1-R-small       WB_temperature.json      1.056907 1.457730e+00      1.207365  1.867155    0.111371    0.016186     0.127225 0.031413
   WB90             ph Salesforce/moirai-1.1-R-small                WB_ph.json      0.087265 1.092985e-02      0.104546  1.219065    0.272705    0.106737     0.326706 0.007916
   WB90       humidity Salesforce/moirai-1.1-R-small          WB_humidity.json      

In [18]:
print('\n')
print('=' * 90)
print('PROCESS FINISHED')
print('=' * 90)
print('Datasets requested:', len(DATASETS))
print('Targets requested:', total_models)
print('Targets completed:', len(all_results))
print('Output directory:', OUTPUT_DIR)
if len(all_results) != total_models:
    print('\n⚠️ WARNING:')
    print('Not all requested targets completed successfully.')
    print('Requested:', total_models)
    print('Completed:', len(all_results))
else:
    print('\n✔️ ALL REQUESTED TARGETS COMPLETED.')



PROCESS FINISHED
Datasets requested: 6
Targets requested: 26
Targets completed: 26
Output directory: /content/drive/MyDrive/results/json/Moirai

✔️ ALL REQUESTED TARGETS COMPLETED.
